# Data recipes — every pull, in one place

A working reference for the five data families the sweep research uses: **sweeps**,
**golden sweeps**, **ticker history**, **earnings**, and **FOMC**. Every cell runs, so
copy from here rather than re-deriving a call.

For what the API contains, see
[`quantdata/explore_api.ipynb`](../quantdata/explore_api.ipynb). For the strategy
itself, [`directional_sweeps.ipynb`](directional_sweeps.ipynb) and
[`directional_backtest.ipynb`](directional_backtest.ipynb).

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd() / "research" / "quantdata"))
sys.path.insert(0, str(Path.cwd() / "research" / "sweep_events"))

import feeds as F          # all API pulls
import directional as D    # cached flow + signal features
import earnings as E       # earnings calendar
import fomc                # FOMC decision days

pd.set_option("display.width", 150, "display.max_columns", 40)

## Which backend serves what

Three lanes. `data-routing` defines the first two; Quant Data is a third, read-only
historical lane for what neither of the others carries.

| Want | Call | Notes |
|---|---|---|
| Option sweep prints | `F.sweeps(...)` | only Quant Data has these |
| One contract's option bars | `F.option_bars(...)` | only Quant Data has these |
| One ticker, intraday, one session | `F.stock_bars(...)` | **unadjusted** — matches strikes |
| Many tickers, years, daily | `F.alpaca_bars(...)` | **split-adjusted**, cached, bulk |
| Earnings dates | `E.frame()` / `E.next_in_window(...)` | Robinhood-sourced parquet |
| FOMC decision days | `fomc.frame()` / `fomc.is_decision_day(...)` | Fed calendar, 2025-2027 |

**The adjustment trap.** Quant Data prices are unadjusted; Alpaca's are
split-adjusted. A strike is unadjusted, so comparing it to an Alpaca close across a
split is what produced the fictional CRWD +5,368%. Use `stock_bars` where a price
meets a strike, `alpaca_bars` for return series and indicators.

## 1. Sweeps

`F.sweeps()` defaults to the strategy's screen — consolidation `SWEEP`, type `ISO`,
premium >= $100k — and every filter is applied server-side.

In [2]:
one_day = F.sweeps("2026-09-21")
print(f"{len(one_day)} ISO sweeps >= $100k on 2026-09-21, {one_day.ticker.nunique()} tickers")
one_day.head(4)[["ts", "ticker", "contractType", "strikePrice", "expirationDate",
                 "optionPrice", "size", "premium", "tradeSideCode", "greeks.delta"]]

421 ISO sweeps >= $100k on 2026-09-21, 93 tickers


,ts,ticker,contractType,strikePrice,expirationDate,optionPrice,size,premium,tradeSideCode,greeks.delta
0,2026-09-21 16:09:12.663000-04:00,QQQ,PUT,730.0,2026-10-16,8.52,228,194256.0,BID,-0.342392
1,2026-09-21 16:08:49.862000-04:00,SPY,PUT,773.0,2026-09-22,1.45,1000,145344.0,BELOW_BID,-0.450595
2,2026-09-21 15:59:54.352000-04:00,SPCX,CALL,155.0,2026-09-25,2.32,500,116051.0,ASK,0.379496
3,2026-09-21 15:59:52.675000-04:00,TE,CALL,5.0,2026-10-02,0.20,5000,100000.0,ASK,0.343038


### Setting filters

In [3]:
# premium floor
big = F.sweeps("2026-09-21", premium_min=1_000_000)

# specific tickers
nv = F.sweeps("2026-09-21", tickers=["NVDA", "TSLA"])

# calls only, bought at or above the offer (aggressive buyers)
lifted = F.sweeps("2026-09-21", premium_min=500_000,
                  contract_types=["CALL"], sides=["ASK", "ABOVE_ASK"])

# drop the ISO restriction entirely
any_type = F.sweeps("2026-09-21", premium_min=1_000_000, trade_types=None)

for label, frame in [("premium >= $1M", big), ("NVDA+TSLA", nv),
                     ("calls lifted >= $500k", lifted), ("any trade type >= $1M", any_type)]:
    print(f"  {label:24} {len(frame):4} prints")

  premium >= $1M             13 prints
  NVDA+TSLA                  46 prints
  calls lifted >= $500k      11 prints
  any trade type >= $1M     443 prints


Filters the API does not expose are applied after the pull — delta band is one, since
the greeks are per-print rather than a query field:

In [4]:
band = big[big["greeks.delta"].abs().between(0.30, 0.70)]
print(f"delta 0.30-0.70: {len(band)} of {len(big)}")

# a date range loops sessions; pass progress=True to watch it
week = F.sweeps(start="2026-09-14", end="2026-09-18", premium_min=1_000_000)
print(f"week of 14 Sep: {len(week)} prints across {week.session.nunique()} sessions")

delta 0.30-0.70: 8 of 13


week of 14 Sep: 58 prints across 5 sessions


### Use the cache for anything repeated

A range pull is one paginated call chain per session. For repeated work, cache once
and read parquet:

```bash
python research/sweep_events/fetch_flow.py 2026-06-01 2026-09-21   # incremental
```

In [5]:
flow = D.load()
print(f"cached: {len(flow):,} prints, {flow.ticker.nunique()} tickers, "
      f"{flow.session.nunique()} sessions ({flow.session.min()} .. {flow.session.max()})")
print(f"columns: {len(flow.columns)} — direction and signed_premium are added on load")

cached: 25,610 prints, 853 tickers, 78 sessions (2026-06-01 .. 2026-09-21)
columns: 34 — direction and signed_premium are added on load


## 2. Golden sweeps — and why the badge is not the screen

The vendor sets its own `isGoldenSweep` flag. It is **a different population from
ISO + SWEEP, not a subset**, and the difference matters for the strategy's premise.

In [6]:
iso = F.sweeps("2026-09-21", premium_min=1_000_000)
gold = F.sweeps("2026-09-21", premium_min=1_000_000, trade_types=None, golden=True)
both = set(iso.id) & set(gold.id)

print(f"ISO + SWEEP   >= $1M : {len(iso):4}")
print(f"isGoldenSweep >= $1M : {len(gold):4}")
print(f"in both              : {len(both):4}")
print()
print("trade types among golden-flagged prints that are NOT ISO:")
print(gold[~gold.id.isin(iso.id)].tradeType.value_counts().to_string())

ISO + SWEEP   >= $1M :   13
isGoldenSweep >= $1M :  121
in both              :    9

trade types among golden-flagged prints that are NOT ISO:
tradeType
AUTO              70
MULTI_AUTO_COB    40
MULTI_AUCT_COB     2


Seventy of the golden-only prints are **`AUTO`** — ordinary electronic fills, not
intermarket sweeps — and forty-two are `MULTI_*_COB`, which are multi-leg and so
directionally ambiguous (`papers/quantdata_trade_types.md` says to exclude those).
Four ISO sweeps carry no badge at all.

So the badge is a loose "unusual activity" marker, while **`ISO` is a statement about
how the order routed**. The strategy's premise — that the print swept multiple
exchanges — is the ISO condition. `F.golden_sweeps()` keeps the ISO default for that
reason; pass `trade_types=None` only if you deliberately want the vendor's badge alone.

In [7]:
strict = F.golden_sweeps("2026-09-21")        # badge AND ISO — the conservative read
print(f"golden AND ISO, >= $1M: {len(strict)} prints")
strict[["ts", "ticker", "contractType", "strikePrice", "premium", "tradeType",
        "tradeSideCode", "isGoldenSweep"]]

golden AND ISO, >= $1M: 9 prints


,ts,ticker,contractType,strikePrice,premium,tradeType,tradeSideCode,isGoldenSweep
0,2026-09-21 15:57:28.698000-04:00,TSLA,PUT,380.0,1734940.0,ISO,BELOW_BID,True
1,2026-09-21 13:51:34.386000-04:00,MSFT,PUT,497.5,1325660.0,ISO,BID,True
2,2026-09-21 13:34:04.001000-04:00,SNDK,PUT,1900.0,3848140.0,ISO,BID,True
3,2026-09-21 13:29:40.494000-04:00,AAPL,PUT,340.0,1524695.0,ISO,BID,True
4,2026-09-21 13:16:08.472000-04:00,META,PUT,680.0,1634632.0,ISO,BID,True
5,2026-09-21 13:13:38.937000-04:00,SPXS,PUT,31.0,1184790.0,ISO,BID,True
6,2026-09-21 11:05:21.480000-04:00,HUT,CALL,105.0,9994876.0,ISO,ABOVE_ASK,True
7,2026-09-21 10:29:44.322000-04:00,SPY,CALL,635.0,1245362.0,ISO,BELOW_BID,True
8,2026-09-21 09:38:23.708000-04:00,SNDK,PUT,1705.0,1442070.0,ISO,BID,True


## 3. Ticker history

Two sources, and the choice is about **price adjustment**, not convenience.

In [8]:
# Quant Data — one ticker, one session, intraday, UNADJUSTED.
bars = F.stock_bars("NVDA", "2026-09-21", period="ONE_HOUR")
print("stock_bars (unadjusted, matches strikes):")
print(bars.head(3).to_string(index=False))
print(f"\nperiods available: {', '.join(F.PERIODS)}")

stock_bars (unadjusted, matches strikes):
                       ts     timestamp  openPrice  highPrice  lowPrice  closePrice
2026-09-21 09:00:00-04:00 1789995600000     222.96     224.91    221.72      224.19
2026-09-21 10:00:00-04:00 1789999200000     224.19     225.14    222.59      224.97
2026-09-21 11:00:00-04:00 1790002800000     224.97     225.48    224.71      225.11

periods available: ONE_MINUTE, FIVE_MINUTE, TEN_MINUTE, FIFTEEN_MINUTE, THIRTY_MINUTE, ONE_HOUR, TWO_HOUR, FOUR_HOUR, ONE_DAY


In [9]:
# Alpaca — bulk, multi-year, multi-symbol, SPLIT-ADJUSTED. The repo's historical lane.
panel = F.alpaca_bars(["NVDA", "TSLA"], "2026-06-01", "2026-09-21", timeframe="1Day")
daily = panel.frame
print(f"alpaca_bars: {daily.shape} — {type(panel).__name__}, use .frame")
print(daily.tail(3).to_string())

alpaca_bars: (156, 7) — BarPanel, use .frame
                                     open    high      low   close       volume  trade_count        vwap
timestamp                 symbol                                                                        
2026-09-18 00:00:00-04:00 TSLA    369.000  370.90  360.751  364.27   52171760.0     891277.0  364.924130
2026-09-21 00:00:00-04:00 NVDA    222.935  228.50  221.560  227.38  111548034.0    3113656.0  226.000137
                          TSLA    371.635  378.36  371.070  375.30   36880559.0     934959.0  375.145153


### One option contract's bars

Only Quant Data serves these. One call per contract-session — a multi-day hold is one
call per day.

In [10]:
opt = F.option_bars("TSLA", "2026-10-16", 360, "CALL", "2026-09-21", period="ONE_HOUR")
print(f"TSLA 10/16 360C on 2026-09-21: {len(opt)} bars, {int(opt.volume.sum())} contracts")
print(opt.to_string(index=False))

TSLA 10/16 360C on 2026-09-21: 7 bars, 961 contracts
                       ts     timestamp  openPrice  highPrice  lowPrice  closePrice  volume
2026-09-21 09:00:00-04:00 1789995600000      22.00      27.75     22.00       27.50     322
2026-09-21 10:00:00-04:00 1789999200000      26.87      27.75     25.20       26.16     499
2026-09-21 11:00:00-04:00 1790002800000      25.82      25.82     24.61       25.65      33
2026-09-21 12:00:00-04:00 1790006400000      24.85      25.50     24.20       24.20      13
2026-09-21 13:00:00-04:00 1790010000000      24.15      24.15     23.57       23.72      20
2026-09-21 14:00:00-04:00 1790013600000      24.20      26.00     24.20       26.00      22
2026-09-21 15:00:00-04:00 1790017200000      25.55      26.20     24.41       25.15      52


An empty frame means the contract **did not trade**, which is information rather than
an error — bar density is a liquidity measure. It also looks identical to a contract
that was never listed, which is how the Juneteenth expiry bug hid: June 2026's third
Friday is the 19th, a holiday, so the monthly was the 18th.

Discover expiries and strikes from what actually traded instead of computing them:

In [11]:
monthlies = D.monthly_expiries(flow)
print("listed monthlies:", [m.date().isoformat() for m in monthlies[:7]])
print("next monthly after 2026-06-02:", D.next_monthly(pd.Timestamp("2026-06-02"), monthlies).date())
print()
traded = flow[(flow.ticker == "NVDA") & (flow.expirationDate == "2026-10-16")]
print("NVDA 10/16 strikes seen in flow:", sorted(traded.strikePrice.unique())[:10])

listed monthlies: ['2026-06-18', '2026-07-17', '2026-08-21', '2026-09-18', '2026-10-16', '2026-11-20', '2026-12-18']
next monthly after 2026-06-02: 2026-06-18

NVDA 10/16 strikes seen in flow: [150.0, 170.0, 180.0, 190.0, 200.0, 205.0, 210.0, 215.0, 220.0, 225.0]


## 4. Earnings

`data/earnings_dates.parquet` — 27,465 reports across 5,788 symbols, Robinhood-sourced.
`timing` is `am`/`pm`; `verified` separates a confirmed date from a projection.

In [12]:
cal = E.frame()
print(f"{len(cal):,} reports, {cal.symbol.nunique():,} symbols, "
      f"{cal.date.min():%Y-%m-%d} .. {cal.date.max():%Y-%m-%d}")
print()
print("NVDA:")
print(cal[cal.symbol == "NVDA"][["date", "timing", "verified", "eps_estimate", "eps_actual"]]
      .to_string(index=False))

27,465 reports, 5,788 symbols, 2025-10-01 .. 2026-12-20

NVDA:
      date timing  verified eps_estimate eps_actual
2025-11-19     pm      True     1.220000   1.300000
2026-02-25     pm      True     1.500000   1.620000
2026-05-20     pm      True     1.760000   1.870000
2026-08-26     pm      True     2.090000   2.220000
2026-11-17     pm      True     2.340000       None


In [13]:
# the veto: does a report land between entry and expiry?
hit = E.next_in_window("NVDA", "2026-06-02", "2026-06-18 16:00", cal)
print(f"NVDA earnings between 2 Jun and 18 Jun expiry : {hit}")
print(f"NVDA earnings between 2 Jun and 21 Aug expiry : "
      f"{E.next_in_window('NVDA', '2026-06-02', '2026-08-21 16:00', cal)}")
print()
# require() raises on a symbol with no coverage — the hand-typed dict passed silently
try:
    E.require(["NVDA", "NOT_A_TICKER"], cal)
except E.NoCoverage as exc:
    print(f"NoCoverage: {exc}")

NVDA earnings between 2 Jun and 18 Jun expiry : None
NVDA earnings between 2 Jun and 21 Aug expiry : None

NoCoverage: no earnings calendar for ['NOT_A_TICKER'] — re-pull the windows and re-run ingest(); these would otherwise pass the veto silently


To refresh: ask Claude to re-pull `get_earnings_calendar` in 31-day windows, then

```bash
python research/sweep_events/earnings.py <session tool-results dir>
```

The MCP tool is callable from Claude, not from Python, and it answers market-wide, so
this is the one calendar that cannot regenerate itself unattended.

## 5. FOMC

`fomc.py` carries every scheduled decision, 2025-2027, from federalreserve.gov. Only
the **second day** of each two-day meeting is an event — the statement lands 14:00 ET.

In [14]:
days = fomc.frame()
print(f"{len(days)} decisions, {int(days.sep.sum())} with SEP (dot plot)")
print(days[days.decision_date.dt.year == 2026].to_string(index=False))

24 decisions, 12 with SEP (dot plot)
decision_date   sep        statement_at  year
   2026-01-28 False 2026-01-28 14:00:00  2026
   2026-03-18  True 2026-03-18 14:00:00  2026
   2026-04-29 False 2026-04-29 14:00:00  2026
   2026-06-17  True 2026-06-17 14:00:00  2026
   2026-07-29 False 2026-07-29 14:00:00  2026
   2026-09-16  True 2026-09-16 14:00:00  2026
   2026-10-28 False 2026-10-28 14:00:00  2026
   2026-12-09  True 2026-12-09 14:00:00  2026


In [15]:
print(f"is 2026-09-16 a decision day? {fomc.is_decision_day('2026-09-16')}")
print(f"is 2026-09-15 (day one)?      {fomc.is_decision_day('2026-09-15')}")
print(f"09:45 on decision day is pre-statement? {fomc.before_statement('2026-09-16 09:45')}")
print(f"15:30 on decision day?                  {fomc.before_statement('2026-09-16 15:30')}")
print()
print(f"first decision between 9 Sep and 16 Oct expiry: "
      f"{fomc.next_in_window('2026-09-09 15:48', '2026-10-16 16:00')}")
print(f"decisions spanned by a 1 Jun - 31 Dec hold:     "
      f"{fomc.count_in_window('2026-06-01', '2026-12-31')}")

is 2026-09-16 a decision day? True
is 2026-09-15 (day one)?      False
09:45 on decision day is pre-statement? True
15:30 on decision day?                  False

first decision between 9 Sep and 16 Oct expiry: 2026-09-16
decisions spanned by a 1 Jun - 31 Dec hold:     5


## 6. Putting it together

The screen, both skips, and the features, end to end:

In [16]:
trig = D.triggers(flow)                    # ISO + SWEEP + $1M + delta band, single names
conf = D.confirm(flow, trig)               # point-in-time confirmation features
cal = E.frame()

# skip 1 — the sweep printed on an FOMC decision day
conf = conf[~conf.entry_ts.map(fomc.is_decision_day)]

# skip 2 — a report lands between entry and the monthly expiry
monthlies = D.monthly_expiries(flow)
conf["expiry"] = conf.entry_ts.map(lambda t: D.next_monthly(t, monthlies))
conf = conf[conf.expiry.notna()]
conf["earnings"] = [E.next_in_window(t, a, b + pd.Timedelta(hours=16), cal)
                    if t in set(cal.symbol) else "no_coverage"
                    for t, a, b in zip(conf.ticker, conf.entry_ts, conf.expiry)]
keep = conf[conf.earnings.isna()]

print(f"{len(trig)} triggers")
print(f"  -> after FOMC-day skip and earnings veto: {len(keep)}")
print()
print("features:", [c for c in keep.columns
                    if c in ("prior_n", "same_dir_n", "opp_dir_n",
                             "net_dir_n", "conviction", "aligned")])
print()
print("priced results live in data/directional_trades.parquet:")
tr = pd.read_parquet("data/directional_trades.parquet")
print(f"  {len(tr)} triggers, {tr.skip.isna().sum()} priced")

441 triggers
  -> after FOMC-day skip and earnings veto: 334

features: ['prior_n', 'same_dir_n', 'opp_dir_n', 'net_dir_n', 'conviction', 'aligned']

priced results live in data/directional_trades.parquet:
  441 triggers, 305 priced
